# Practice Lab: Math Foundations 3 - Transformer Architecture

8 exercises on matmul, attention, and activation functions.

Each exercise gives you the setup and a `# YOUR CODE HERE` stub. Write the code yourself, run it, then expand **Solution** to check your answer.

In [ ]:
!pip install -q numpy torch matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
print('torch:', torch.__version__)

## Exercise 1 - Matmul by hand

**Task:** Compute the matrix product `C = A @ B` for the two given tensors and print it. Check the result against the expected value in the comment.

In [ ]:
A = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])

# YOUR CODE HERE: compute the matrix product C = A @ B and print it
C = ...   # TODO
# print(C)
# Expected: tensor([[4., 5.], [10., 11.]])

<details><summary>Hint</summary>

The `@` operator does matrix multiplication in PyTorch: `A @ B`. A (2x3) times B (3x2) gives a (2x2) result.
</details>

**Criteria:** `C` holds `A @ B`; printed output matches `tensor([[4., 5.], [10., 11.]])`.

<details><summary>Solution</summary>

In [ ]:
A = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
C = A @ B
print(C)
# Expected: tensor([[4., 5.], [10., 11.]])

</details>

---

## Exercise 2 - Identity QK^T

**Task:** With `Q` and `K` both the 4x4 identity, compute the scaled attention scores `Q @ K.T / sqrt(d)`, then softmax over the last dimension. Print the weights rounded to 3 decimals.

In [ ]:
Q = torch.eye(4)
K = torch.eye(4)

# YOUR CODE HERE: scaled dot-product scores, then softmax over the last dim
scores = ...    # TODO: Q @ K.T / sqrt(d)
weights = ...   # TODO: softmax over dim=-1
# print(weights.round(decimals=3))

<details><summary>Hint</summary>

`d = 4`, so divide by `math.sqrt(4)`. Use `torch.softmax(scores, dim=-1)`. Because Q and K are identical rows, each query attends slightly more to its own position.
</details>

**Criteria:** `scores = Q @ K.T / sqrt(4)`; `weights` softmaxed over `dim=-1` and printed.

<details><summary>Solution</summary>

In [ ]:
Q = torch.eye(4)
K = torch.eye(4)
scores = Q @ K.T / math.sqrt(4)
weights = torch.softmax(scores, dim=-1)
print(weights.round(decimals=3))

</details>

---

## Exercise 3 - Linear collapse

**Task:** Show that two stacked linear layers equal the single combined matrix `W2 @ W1`, then show that inserting a ReLU between them breaks that equivalence (no single matrix reproduces it).

In [ ]:
import numpy as np
W1 = np.array([[1, 0], [1, 1]])
W2 = np.array([[2, 1], [0, 1]])
x  = np.array([1, 2])

# YOUR CODE HERE
# 1) Two linear layers vs the single combined matrix W2 @ W1
# print('two layers:', ...)     # expected [5 3]
# print('combined:  ', ...)     # expected [5 3]  - identical

# 2) Insert a ReLU between them, with x2 = [1, -3]
x2 = np.array([1, -3])
# h = ...                       # ReLU clips the negative: np.maximum(0, W1 @ x2)
# print('with ReLU: ', ...)     # expected [2 0]
# print('collapsed: ', ...)     # expected [0 -2]  - no single matrix matches

<details><summary>Hint</summary>

Apply layers right-to-left: `W2 @ (W1 @ x)`. ReLU is `np.maximum(0, ...)`. The point: without a nonlinearity, stacked linear layers collapse to one matrix.
</details>

**Criteria:** two-layer output equals combined for `x`; with ReLU on `x2`, the two outputs differ.

<details><summary>Solution</summary>

In [ ]:
import numpy as np
W1 = np.array([[1, 0], [1, 1]])
W2 = np.array([[2, 1], [0, 1]])
x  = np.array([1, 2])

# Two linear layers vs the single combined matrix W2 @ W1
print('two layers:', W2 @ (W1 @ x))     # [5 3]
print('combined:  ', (W2 @ W1) @ x)     # [5 3]  - identical

# Insert a ReLU between them, with x = [1, -3]
x2 = np.array([1, -3])
h  = np.maximum(0, W1 @ x2)              # ReLU clips the negative
print('with ReLU: ', W2 @ h)            # [2 0]
print('collapsed: ', (W2 @ W1) @ x2)    # [0 -2]  - no single matrix matches

</details>

---

## Exercise 4 - Causal mask correctness

**Task:** Build causal (lower-triangular) attention weights. Compute scaled scores, mask the upper triangle with `-inf`, softmax, and assert each row only attends to positions at or before it.

In [ ]:
torch.manual_seed(0)
N, d = 5, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

# YOUR CODE HERE: build causal (lower-triangular) attention weights
scores = ...   # TODO: Q @ K.T / sqrt(d)
# TODO: mask out the upper triangle with -inf, then softmax over dim=-1
weights = ...
# print(weights.round(decimals=3))
# assert (weights.triu(diagonal=1) == 0).all()

<details><summary>Hint</summary>

`mask = torch.tril(torch.ones(N, N))` keeps the lower triangle. Use `scores.masked_fill(mask == 0, float('-inf'))` before softmax so masked positions get zero weight.
</details>

**Criteria:** upper-triangular weights are all 0; the `assert` passes.

<details><summary>Solution</summary>

In [ ]:
torch.manual_seed(0)
N, d = 5, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

scores = Q @ K.T / math.sqrt(d)
mask = torch.tril(torch.ones(N, N))
scores = scores.masked_fill(mask == 0, float('-inf'))
weights = torch.softmax(scores, dim=-1)
print(weights.round(decimals=3))
assert (weights.triu(diagonal=1) == 0).all()

</details>

---

## Exercise 5 - sqrt(d_k) variance check

**Task:** For each `d_k`, compute the raw scores `Q @ K.T` and the `sqrt(d_k)`-scaled version, and print both variances. Confirm scaling holds variance near 1 while raw variance grows with `d_k`.

In [ ]:
for d_k in [4, 16, 64, 256]:
    Q = torch.randn(1000, d_k)
    K = torch.randn(1000, d_k)
    # YOUR CODE HERE: raw scores and the sqrt(d_k)-scaled version
    raw = ...      # TODO: Q @ K.T
    scaled = ...   # TODO: raw / sqrt(d_k)
    # print(f'd_k={d_k:4d}  raw_var={raw.var().item():.2f}  scaled_var={scaled.var().item():.2f}')

<details><summary>Hint</summary>

Dividing by `math.sqrt(d_k)` divides the variance by `d_k`. Raw variance scales roughly linearly with `d_k`; scaled variance stays close to 1.
</details>

**Criteria:** raw variance grows with `d_k`; scaled variance stays near 1 across all four values.

<details><summary>Solution</summary>

In [ ]:
for d_k in [4, 16, 64, 256]:
    Q = torch.randn(1000, d_k)
    K = torch.randn(1000, d_k)
    raw = (Q @ K.T)
    scaled = raw / math.sqrt(d_k)
    print(f'd_k={d_k:4d}  raw_var={raw.var().item():.2f}  scaled_var={scaled.var().item():.2f}')

</details>

---

## Exercise 6 - Activation comparison

**Task:** Compute the ReLU, GELU, and SiLU activations of `x`. The plot and per-value print loop are provided as a check - fill in the three activation vectors so they render.

In [ ]:
x = torch.linspace(-3, 3, 200)
# YOUR CODE HERE: compute the three activations of x
relu = ...   # TODO: F.relu(x)
gelu = ...   # TODO: F.gelu(x)
silu = ...   # TODO: F.silu(x)
plt.plot(x.numpy(), relu.numpy(), label='ReLU', color='red')
plt.plot(x.numpy(), gelu.numpy(), label='GELU', color='blue')
plt.plot(x.numpy(), silu.numpy(), label='SiLU (swish)', color='purple')
plt.legend(); plt.title('Activations'); plt.show()
for xv in [-1.0, 0.0, 1.0]:
    t = torch.tensor([xv])
    print(f'x={xv:+.1f}  ReLU={F.relu(t).item():.3f}  GELU={F.gelu(t).item():.3f}  SiLU={F.silu(t).item():.3f}')

<details><summary>Hint</summary>

Use `F.relu(x)`, `F.gelu(x)`, and `F.silu(x)` (SiLU is also called swish). GELU and SiLU are smooth near 0; ReLU has a hard corner.
</details>

**Criteria:** all three curves render; the print loop shows GELU and SiLU are smooth while ReLU zeros negatives.

<details><summary>Solution</summary>

In [ ]:
x = torch.linspace(-3, 3, 200)
relu = F.relu(x); gelu = F.gelu(x); silu = F.silu(x)
plt.plot(x.numpy(), relu.numpy(), label='ReLU', color='red')
plt.plot(x.numpy(), gelu.numpy(), label='GELU', color='blue')
plt.plot(x.numpy(), silu.numpy(), label='SiLU (swish)', color='purple')
plt.legend(); plt.title('Activations'); plt.show()
for xv in [-1.0, 0.0, 1.0]:
    t = torch.tensor([xv])
    print(f'x={xv:+.1f}  ReLU={F.relu(t).item():.3f}  GELU={F.gelu(t).item():.3f}  SiLU={F.silu(t).item():.3f}')

</details>

---

## Exercise 7 - Implement scaled dot-product attention

**Task:** Fill in `my_sdpa` so it matches `F.scaled_dot_product_attention`. Compute scaled scores, apply the optional mask, softmax, and weight `V`. The harness checks your output against the reference.

In [ ]:
def my_sdpa(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    # YOUR CODE HERE: implement scaled dot-product attention
    # 1) scores = Q @ K^T / sqrt(d_k)
    # 2) if mask is not None: fill masked positions (mask == 0) with -inf
    # 3) softmax over the last dim, then return weights @ V
    ...  # TODO
    return ...  # TODO

torch.manual_seed(0)
Q = torch.randn(4, 8)
K = torch.randn(4, 8)
V = torch.randn(4, 8)
mine = my_sdpa(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)
print(f'max abs diff: {(mine - ref).abs().max().item():.2e}')
assert torch.allclose(mine, ref, atol=1e-5)

<details><summary>Hint</summary>

Transpose the last two dims of K with `K.transpose(-2, -1)`. Softmax over `dim=-1`, then matrix-multiply the weights by `V`. Keep the `mask == 0 -> -inf` branch so causal masks work too.
</details>

**Criteria:** `max abs diff` is around 1e-7; the `assert torch.allclose(...)` passes.

<details><summary>Solution</summary>

In [ ]:
def my_sdpa(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(4, 8)
K = torch.randn(4, 8)
V = torch.randn(4, 8)
mine = my_sdpa(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)
print(f'max abs diff: {(mine - ref).abs().max().item():.2e}')
assert torch.allclose(mine, ref, atol=1e-5)

</details>

---

## Exercise 8 - Multi-head extension

**Task:** Fill in `my_multihead`: project `x` to qkv, split into heads, run attention per head, concat, and apply the output projection. The harness checks the output shape is `(B, T, D)`.

In [ ]:
def my_multihead(x, qkv_weight, out_weight, n_heads):
    B, T, D = x.shape
    d_head = D // n_heads
    # YOUR CODE HERE: implement multi-head self-attention
    # 1) project x to qkv (x @ qkv_weight.T), split into q, k, v with .chunk(3, dim=-1)
    # 2) reshape each to (B, n_heads, T, d_head): .view(B, T, n_heads, d_head).transpose(1, 2)
    # 3) scaled dot-product attention per head, then softmax over dim=-1
    # 4) concat heads back to (B, T, D) and apply the output projection (@ out_weight.T)
    ...  # TODO
    return ...  # TODO

torch.manual_seed(0)
B, T, D, H = 2, 6, 32, 4
x = torch.randn(B, T, D)
qkv_w = torch.randn(3*D, D)
out_w = torch.randn(D, D)
result = my_multihead(x, qkv_w, out_w, H)
print(f'output shape: {result.shape}')
assert result.shape == (B, T, D)

<details><summary>Hint</summary>

After per-head attention, undo the head split with `.transpose(1, 2).contiguous().view(B, T, D)` before the output projection. This is Exercise 7 done in parallel across `n_heads`.
</details>

**Criteria:** `output shape` prints `torch.Size([2, 6, 32])`; the `assert` passes.

<details><summary>Solution</summary>

In [ ]:
def my_multihead(x, qkv_weight, out_weight, n_heads):
    B, T, D = x.shape
    d_head = D // n_heads
    qkv = x @ qkv_weight.T
    q, k, v = qkv.chunk(3, dim=-1)
    q = q.view(B, T, n_heads, d_head).transpose(1, 2)
    k = k.view(B, T, n_heads, d_head).transpose(1, 2)
    v = v.view(B, T, n_heads, d_head).transpose(1, 2)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=-1)
    attn = weights @ v
    out = attn.transpose(1, 2).contiguous().view(B, T, D)
    return out @ out_weight.T

torch.manual_seed(0)
B, T, D, H = 2, 6, 32, 4
x = torch.randn(B, T, D)
qkv_w = torch.randn(3*D, D)
out_w = torch.randn(D, D)
result = my_multihead(x, qkv_w, out_w, H)
print(f'output shape: {result.shape}')
assert result.shape == (B, T, D)

</details>

---

## Wrap-up

Exercise 4 (causal mask) and Exercise 7 (single-head from scratch) make the math click. Lesson 1.4 adds the plumbing.